# M02 单语义特征与字典学习


## Sparse autoencoder 教学实验

我们先用一个隐藏字典生成激活，再训练一个小型 sparse autoencoder 去恢复可复用方向。


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(11)

activation_dim = 6
true_features = 4
sae_features = 8

true_dictionary = torch.tensor(
    [
        [1.0, 0.0, 0.0, 0.5],
        [0.8, 0.1, 0.2, 0.0],
        [0.0, 0.9, 0.1, 0.4],
        [0.0, 0.8, 0.4, 0.1],
        [0.3, 0.0, 0.9, 0.2],
        [0.1, 0.2, 0.8, 1.0],
    ],
    dtype=torch.float32,
)

encoder = torch.nn.Linear(activation_dim, sae_features)
decoder = torch.nn.Linear(sae_features, activation_dim, bias=False)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.03)

for step in range(700):
    sparse_codes = (torch.rand(256, true_features) < 0.28).float() * torch.rand(256, true_features)
    activations = sparse_codes @ true_dictionary.T + 0.02 * torch.randn(256, activation_dim)
    hidden = torch.relu(encoder(activations))
    recon = decoder(hidden)
    loss = torch.nn.functional.mse_loss(recon, activations) + 0.01 * hidden.mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

decoder_weights = decoder.weight.detach().T
norms = decoder_weights.norm(dim=1)
top_indices = torch.topk(norms, k=4).indices

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(true_dictionary, cmap="viridis", aspect="auto")
axes[0].set_title("Ground-truth dictionary")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Activation dimension")

axes[1].imshow(decoder_weights[top_indices], cmap="viridis", aspect="auto")
axes[1].set_title("Recovered decoder directions (top 4)")
axes[1].set_xlabel("Activation dimension")
axes[1].set_ylabel("Recovered feature")
plt.tight_layout()

print("Top recovered feature indices:", top_indices.tolist())
print("Final loss:", float(loss.detach()))


## 小结

恢复出来的方向不会完美等于我们埋进去的字典，但它们已经比原始神经元更可复用、更稀疏，也更容易讲清楚。
